pancreas dataset from https://figshare.com/articles/dataset/scIB_pancreas_dataset/25953868

In [29]:
import scanpy as sc
import anndata
import pytorch_lightning as pl  
from scdiff.dataset import ScDataModule
from scdiff.ae import LightningAE
from scdiff.utils import compare_umap
import os

In [30]:
pancreas = anndata.read_h5ad('../data/scIBPancreas.h5ad')
display(pancreas.X)
display(pancreas.obs)

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [2.6120808 , 0.        , 0.        , ..., 0.        , 2.6120806 ,
        0.        ],
       [0.        , 3.311074  , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.42996258, 2.6206095 , 0.        , ..., 2.1124895 , 1.0953737 ,
        1.0403827 ],
       [3.4695568 , 0.64595073, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.4566651 , 0.5159923 , 0.        , ..., 0.        , 0.53929645,
        0.        ]], shape=(16382, 19093), dtype=float32)

,tech,celltype,size_factors
D101_5,celseq,gamma,0.028492
D101_43,celseq,gamma,0.079348
D101_93,celseq,gamma,0.037932
D102_4,celseq,gamma,0.047685
D172444_23,celseq,gamma,0.038683
...,...,...,...
Sample_1594,smarter,gamma,1.000000
Sample_1595,smarter,gamma,1.000000
Sample_1597,smarter,gamma,1.000000
Sample_1598,smarter,gamma,1.000000


In [ ]:
pancreas.obs['tech'].unique()


['celseq', 'celseq2', 'fluidigmc1', 'smartseq2', 'inDrop1', 'inDrop2', 'inDrop3', 'inDrop4', 'smarter']
Categories (9, object): ['celseq', 'celseq2', 'fluidigmc1', 'inDrop1', ..., 'inDrop3', 'inDrop4', 'smarter', 'smartseq2']

In [ ]:
pancreasModule = ScDataModule(pancreas, "celltype", "LabelEncoder")
ae = LightningAE(n_genes=pancreas.X.shape[1])

log_dir = "../lightning_logs/pancreas/ae"
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

trainer = pl.Trainer(
    max_epochs=50,
    accelerator="auto",
    devices="auto",
    log_every_n_steps=50,
    enable_checkpointing=True,
    logger=True,
    default_root_dir=log_dir
)

ae_path = "checkpoints/pancreas/trained_ae.ckpt"
if os.path.exists(ae_path):
    ae = LightningAE.load_from_checkpoint(ae_path)
else:
    trainer.fit(ae, pancreasModule)
    trainer.save_checkpoint(ae_path)


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs

  | Name      | Type    | Params | Mode 
----------------------------------------------
0 | encoder   | Encoder | 21.8 M | train
1 | decoder   | Decoder | 21.8 M | train
2 | criterion | MSELoss | 0      | train
----------------------------------------------
43.6 M    Trainable params
0         Non-trainable params
43.6 M    Total params
174.386   Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


Epoch 39:  36%|███▌      | 37/103 [00:06<00:10,  6.09it/s, v_num=0, val_loss=0.757, train_loss=0.719] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: To exit: use 'exit', 'quit', or Ctrl-D.
